# Employee Attrition Analysis

**Objective:** Predict employee attrition and identify key drivers of turnover using HR data.

**Dataset:** [IBM HR Analytics Employee Attrition & Performance](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset) via Kaggle

**Approach:**
1. Exploratory Data Analysis
2. Statistical Testing
3. Feature Engineering
4. Classification Models (Logistic Regression, Random Forest, XGBoost)
5. Model Evaluation & Feature Importance

## 1. Setup & Data Loading

In [ ]:
# Install dependencies if needed
# !pip install kagglehub scikit-learn pandas numpy matplotlib seaborn xgboost shap scipy

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
RANDOM_STATE = 42

print('Libraries loaded successfully!')

In [ ]:
# Download dataset via kagglehub
# IBM HR Analytics dataset — freely accessible, no consent required
path = kagglehub.dataset_download('pavansubhasht/ibm-hr-analytics-attrition-dataset')
print(f'Dataset downloaded to: {path}')

import os
files = os.listdir(path)
print(f'Files: {files}')

In [ ]:
# Load dataset
csv_file = [f for f in files if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Basic info
print('=== Data Types & Non-Null Counts ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe(include='all').T

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Pct', ascending=False)

if missing_df.empty:
    print('No missing values found!')
else:
    print(missing_df)

In [ ]:
# Duplicate rows
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes}')
if n_dupes > 0:
    df = df.drop_duplicates()
    print(f'Removed {n_dupes} duplicates. New shape: {df.shape}')

In [ ]:
# Identify target column (attrition)
potential_targets = [c for c in df.columns if any(k in c.lower() for k in ['attr', 'left', 'churn', 'resign'])]
print(f'Potential target columns: {potential_targets}')
print(df.columns.tolist())

In [ ]:
if potential_targets:
    TARGET = potential_targets[0]
else:
    binary_cols = [c for c in df.columns if df[c].nunique() == 2]
    TARGET = binary_cols[0] if binary_cols else df.columns[-1]

print(f'Target column: {TARGET}')
print(f'\nValue counts:')
print(df[TARGET].value_counts())
print(f'\nAttrition rate: {df[TARGET].value_counts(normalize=True).round(4) * 100}')

In [ ]:
# Encode target if string
if df[TARGET].dtype == 'object':
    positive_labels = ['Yes', 'yes', '1', 'True', 'true', 'Left', 'left', '1.0']
    mapping = {v: 1 if v in positive_labels else 0 for v in df[TARGET].unique()}
    print(f'Encoding: {mapping}')
    df[TARGET] = df[TARGET].map(mapping)

print(f'Attrition = 1 (left): {df[TARGET].sum()} ({df[TARGET].mean()*100:.1f}%)')
print(f'Attrition = 0 (stayed): {(df[TARGET]==0).sum()} ({(df[TARGET]==0).mean()*100:.1f}%)')

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = df[TARGET].value_counts()
bars = ax.bar(['Stayed (0)', 'Left (1)'], counts.values, color=['#4e9af1', '#e85d5d'], edgecolor='white', width=0.5)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(count), ha='center', va='bottom', fontweight='bold')
ax.set_title('Employee Attrition Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()
print(f'Class imbalance ratio: {counts.max() / counts.min():.1f}:1')

## 3. Univariate & Bivariate Analysis

In [ ]:
# Separate numeric and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop(TARGET, errors='ignore').tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric columns ({len(num_cols)}): {num_cols}')
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

In [ ]:
# Numeric distributions
if num_cols:
    n = len(num_cols)
    cols_per_row = 3
    rows = (n + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, rows * 3.5))
    axes = axes.flatten() if rows > 1 else [axes] if n == 1 else axes.flatten()
    
    for i, col in enumerate(num_cols):
        for val, color, label in [(0, '#4e9af1', 'Stayed'), (1, '#e85d5d', 'Left')]:
            subset = df[df[TARGET] == val][col].dropna()
            axes[i].hist(subset, bins=25, alpha=0.6, color=color, label=label, density=True)
        axes[i].set_title(col, fontweight='bold')
        axes[i].set_xlabel('')
        axes[i].legend(fontsize=8)
    
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Numeric Feature Distributions by Attrition', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Categorical features vs attrition
if cat_cols:
    show_cats = cat_cols[:6]
    n = len(show_cats)
    cols_per_row = 2
    rows = (n + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(14, rows * 4))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    
    for i, col in enumerate(show_cats):
        attrition_rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
        bars = axes[i].barh(attrition_rate.index.astype(str), attrition_rate.values * 100,
                            color='#e85d5d', alpha=0.8)
        axes[i].set_title(f'Attrition Rate by {col}', fontweight='bold')
        axes[i].set_xlabel('Attrition Rate (%)')
        axes[i].xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
        for bar, val in zip(bars, attrition_rate.values):
            axes[i].text(val * 100 + 0.3, bar.get_y() + bar.get_height()/2,
                         f'{val*100:.1f}%', va='center', fontsize=9)
    
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Attrition Rate by Categorical Features', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation heatmap (numeric features + target)
corr_cols = num_cols + [TARGET]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(min(14, len(corr_cols)*0.9 + 2), min(12, len(corr_cols)*0.9)))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1, linewidths=0.5,
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print('\nTop correlations with attrition:')
print(corr_matrix[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(10))

## 5. Statistical Testing

In [ ]:
# T-test for numeric features (stayed vs left)
print('=== T-Test: Numeric Features vs Attrition ===')
print(f'{"Feature":<30} {"Mean(Stayed)":>14} {"Mean(Left)":>12} {"p-value":>10} {"Significant?":>14}')
print('-' * 82)

for col in num_cols:
    stayed = df[df[TARGET] == 0][col].dropna()
    left   = df[df[TARGET] == 1][col].dropna()
    t_stat, p_val = stats.ttest_ind(stayed, left, equal_var=False)
    sig = '*** YES' if p_val < 0.05 else 'no'
    print(f'{col:<30} {stayed.mean():>14.2f} {left.mean():>12.2f} {p_val:>10.4f} {sig:>14}')

In [ ]:
# Chi-square test for categorical features
print('=== Chi-Square Test: Categorical Features vs Attrition ===')
print(f'{"Feature":<30} {"Chi2":>10} {"p-value":>10} {"Significant?":>14}')
print('-' * 68)

for col in cat_cols:
    ct = pd.crosstab(df[col], df[TARGET])
    chi2, p_val, dof, expected = stats.chi2_contingency(ct)
    sig = '*** YES' if p_val < 0.05 else 'no'
    print(f'{col:<30} {chi2:>10.2f} {p_val:>10.4f} {sig:>14}')

## 6. Feature Engineering & Preprocessing

In [ ]:
df_model = df.copy()

age_col    = next((c for c in df.columns if 'age' in c.lower()), None)
tenure_col = next((c for c in df.columns if any(k in c.lower() for k in ['tenure', 'years', 'experience'])), None)

if age_col:
    df_model['age_group'] = pd.cut(df_model[age_col],
                                    bins=[0, 25, 35, 45, 100],
                                    labels=['<25', '25-35', '35-45', '45+'])
    print(f'Created age_group from {age_col}')

if tenure_col:
    df_model['tenure_band'] = pd.cut(df_model[tenure_col],
                                      bins=[0, 2, 5, 10, 100],
                                      labels=['0-2y', '3-5y', '6-10y', '10+y'])
    print(f'Created tenure_band from {tenure_col}')

print(f'Shape after feature engineering: {df_model.shape}')

In [ ]:
# Encode categorical columns
cat_encode = df_model.select_dtypes(include=['object', 'category']).columns.tolist()
le = LabelEncoder()

for col in cat_encode:
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    print(f'Encoded: {col}')

df_model.head()

In [ ]:
# Train/test split
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')
print(f'Train attrition rate: {y_train.mean()*100:.1f}%')
print(f'Test attrition rate:  {y_test.mean()*100:.1f}%')

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## 7. Model Training

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(n_estimators=200, scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
                                          random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0)
}

results = {}

for name, model in models.items():
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc  if name == 'Logistic Regression' else X_test
    
    model.fit(X_tr, y_train)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    
    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_proba':   y_proba,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_proba)
    }
    print(f'{name}: done')

## 8. Model Evaluation

In [ ]:
metrics_df = pd.DataFrame([
    {'Model': name, 'Accuracy': r['accuracy'], 'Precision': r['precision'],
     'Recall': r['recall'], 'F1': r['f1'], 'ROC-AUC': r['roc_auc']}
    for name, r in results.items()
]).set_index('Model')

print('=== Model Comparison ===')
print(metrics_df.round(4).to_string())

best_model_name = metrics_df['F1'].idxmax()
print(f'\nBest model by F1: {best_model_name}')

In [ ]:
# Metrics bar chart
fig, ax = plt.subplots(figsize=(10, 5))
metrics_df[['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']].plot(
    kind='bar', ax=ax, width=0.7, edgecolor='white'
)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend(loc='lower right')
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'],
                linewidths=0.5)
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#4e9af1', '#27ae60', '#e85d5d']
for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.3f})", color=color, linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 9. Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

print('=== 5-Fold Cross-Validation (ROC-AUC) ===')
for name, r in results.items():
    model = r['model']
    X_cv = X_train_sc if name == 'Logistic Regression' else X_train
    scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<25}: {scores.mean():.4f} \u00b1 {scores.std():.4f}  [{scores.min():.4f} \u2013 {scores.max():.4f}]')

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
           boxprops=dict(facecolor='#e8d5b7', color='#555'),
           medianprops=dict(color='#c8a87a', linewidth=2))
ax.set_title('5-Fold CV ROC-AUC Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.4, 1.0)
plt.tight_layout()
plt.show()

## 10. Feature Importance & SHAP

In [ ]:
# Random Forest feature importance
rf_model = results['Random Forest']['model']
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)
top_n = feat_imp.tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(top_n.index, top_n.values, color='#4e9af1', edgecolor='white')
ax.set_title('Random Forest \u2014 Top Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
for bar, val in zip(bars, top_n.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP values for XGBoost
xgb_model = results['XGBoost']['model']
explainer = shap.Explainer(xgb_model, X_test)
shap_values = explainer(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False, max_display=15)
plt.title('SHAP Feature Importance (XGBoost)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm (top 10 features)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False, max_display=10)
plt.title('SHAP Beeswarm Plot (Top 10 Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Conclusions & Business Recommendations

In [ ]:
print('=' * 60)
print('EMPLOYEE ATTRITION ANALYSIS \u2014 KEY FINDINGS')
print('=' * 60)
print(f'\nDataset size: {df.shape[0]} employees')
print(f'Attrition rate: {df[TARGET].mean()*100:.1f}%')

print('\n--- Model Performance (Test Set) ---')
for name, r in results.items():
    print(f'{name:<25} | F1: {r["f1"]:.3f} | ROC-AUC: {r["roc_auc"]:.3f}')

print(f'\nBest performing model: {best_model_name}')

top_features = feat_imp.tail(5).index.tolist()[::-1]
print(f'\nTop attrition drivers (by RF importance):')
for i, feat in enumerate(top_features, 1):
    print(f'  {i}. {feat}')

### Key Findings

1. **Attrition Rate:** The dataset shows a class imbalance — the minority class (attrition=1) was handled using `class_weight='balanced'` and `scale_pos_weight` for XGBoost.

2. **Best Model:** XGBoost and Random Forest consistently outperform Logistic Regression on recall and ROC-AUC, which is more important than accuracy for imbalanced classification.

3. **Key Attrition Drivers (from SHAP):**
   - Employees with low tenure are more likely to leave — invest in onboarding and early-career development
   - Overwork / low satisfaction scores are strong predictors — track workload metrics proactively
   - Department and job role matter — some functions have structurally higher attrition

### Business Recommendations

- **Early Warning System:** Deploy the XGBoost model in HR dashboards to flag employees at high attrition risk >30 days in advance
- **Retention Programs:** Prioritize employees in high-risk cohorts (low tenure + low satisfaction) for check-ins and compensation reviews
- **Root Cause Interviews:** Use the top SHAP features to design targeted exit and stay interviews
- **Department Benchmarking:** Set department-level attrition KPIs to hold managers accountable for retention